<a href="https://colab.research.google.com/github/Rajshree-Hada/Problem-Set-3/blob/main/ps3_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [2]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [5]:
# Step 1.1: build the warehouse

# The first 2,000 customers are West, the rest East, so West is exactly MIN_WEST
region = np.where(np.arange(N_CUSTOMERS) < MIN_WEST, "West", "East")

# n_orders = 1 + NegBin(0.8, 0.12): mean near 7, long right tail
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, N_CUSTOMERS)

# revenue is driven by order count, floored at 50
revenue = np.maximum(REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, N_CUSTOMERS), REV_FLOOR).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(N_CUSTOMERS),
    "region": region,
    "revenue": revenue,
    "signup_date": pd.to_datetime("2022-01-01") + pd.to_timedelta(rng.integers(0, 1000, N_CUSTOMERS), unit="D"),
})

# One orders row per order: np.repeat writes each customer_id once per order (as in Lab 11, Step 2)
orders = pd.DataFrame({
    "customer_id": np.repeat(customers["customer_id"].values, n_orders),
})

# Register both tables with DuckDB
con.register("customers", customers)
con.register("orders", orders)
print(con.sql("SHOW TABLES").df().to_string(index=False))

# Confirm in SQL: row counts and the West sample
print(con.sql("""
    SELECT region, COUNT(*) AS customers, ROUND(AVG(revenue), 2) AS avg_revenue
    FROM customers
    GROUP BY region
    ORDER BY region
""").df().to_string(index=False))

# Confirm in SQL: orders per customer, the long tail
print(con.sql("""
    WITH per_customer AS (
        SELECT customer_id, COUNT(*) AS n
        FROM orders
        GROUP BY customer_id
    )
    SELECT COUNT(*)           AS customers_with_orders,
           ROUND(AVG(n), 2)   AS mean_orders,
           MEDIAN(n)          AS median_orders,
           MAX(n)             AS max_orders
    FROM per_customer
""").df().to_string(index=False))

     name
customers
   orders
region  customers  avg_revenue
  East       1000       823.92
  West       2000       812.17
 customers_with_orders  mean_orders  median_orders  max_orders
                  3000         6.83            4.0          57


The warehouse has 3,000 customers, 2,000 in the West and 1,000 in the East,
and every customer has at least one order. Orders per customer average 6.83, with a median of 4 and a maximum of 57. The mean sits well above the median and the maximum is about eight times the mean, so the distribution has the long right tail the brief asked for. Average revenue is similar across regions (East $823.92, West $812.17), which is expected because region plays no part in how the data was generated. Revenue is built from order count, so customers with many orders are also the high-revenue customers. That link is what a join
fan-out can exploit later.



### Step 1.2: Write both queries (7 pts)

In [9]:
# Step 1.2: the two queries, West customers only

# Un-joined: one row per customer
unjoined = con.sql("""
    SELECT COUNT(*)                  AS rows_used,
           ROUND(AVG(revenue), 2)    AS avg_revenue
    FROM customers
    WHERE region = 'West'
""").df()

# Joined: each customer repeated once per order
joined = con.sql("""
    SELECT COUNT(*)                  AS rows_used,
           ROUND(AVG(c.revenue), 2)  AS avg_revenue
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
""").df()

print("Un-joined (customers only):")
print(unjoined.to_string(index=False))
print("\nJoined (customers x orders):")
print(joined.to_string(index=False))

Un-joined (customers only):
 rows_used  avg_revenue
      2000       812.17

Joined (customers x orders):
 rows_used  avg_revenue
     13440      1242.54


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

The un-joined query averages revenue over 2,000 rows and gives $812.17. The joined query averages it over 13,440 rows and gives 1,242.54, a ratio of 1.53. The un-joined number is the correct average revenue per customer. The unit of
observation for `revenue` is the customer. It is one value per customer, stored once in the `customers` table, and a customer's revenue does not change with how many orders they placed. "Average revenue per customer" therefore means one vote per customer, which is what the un-joined query counts.

The join changes the unit of observation from the customer to the order. Each customer's revenue is copied onto one row per order, so the 2,000 customers become 13,440 rows and a customer with 40 orders is counted 40 times while a customer with 1 order is counted once. The joined average is an order-weighted average. Revenue is built from order count in this data, so the heavily weighted customers are exactly the high-revenue ones, and the average is pulled up from $812.17 to 1,242.54. No new customers were added and no revenue values changed, only the weights did.

The joined number answers the question of the average revenue of the customer behind a randomly chosen order.

### Step 1.3: The rule (10 pts)

In [10]:
# Step 1.3: a check that warns when a join changes the row count

def check_join(base_sql, joined_sql, label):
    """Warn if the joined query returns a different number of rows than the base query."""
    before = con.sql(f"SELECT COUNT(*) FROM ({base_sql})").fetchone()[0]
    after = con.sql(f"SELECT COUNT(*) FROM ({joined_sql})").fetchone()[0]
    if after != before:
        print(f"WARNING {label}: the join changed the rows, {before:,} -> {after:,} ({after / before:.2f}x)")
    else:
        print(f"OK      {label}: {before:,} rows before and after the join")
    return after == before

# The base: one row per West customer
base = "SELECT customer_id, revenue FROM customers WHERE region = 'West'"

# Bad: Step 1.2's join, each customer repeated once per order
bad = """
    SELECT c.customer_id, c.revenue
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

# Good: aggregate orders to one row per customer first, then join
good = """
    SELECT c.customer_id, c.revenue
    FROM customers c
    LEFT JOIN (SELECT customer_id, COUNT(*) AS n_orders
               FROM orders
               GROUP BY customer_id) o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

check_join(base, bad, "customers x orders")
check_join(base, good, "customers x orders (aggregated first)")

# The good query's average should match the un-joined number from Step 1.2
print("Good query avg revenue:", con.sql(f"SELECT ROUND(AVG(revenue), 2) FROM ({good})").fetchone()[0])

WARNING customers x orders: the join changed the rows, 2,000 -> 13,440 (6.72x)
OK      customers x orders (aggregated first): 2,000 rows before and after the join
Good query avg revenue: 812.17


After every join, the row count must equal the left table's row count unless a change is intended. `check_join` runs the query with and without the join and compares `COUNT(*)`. It fired on the customers x orders join (2,000 -> 13,440 rows, 6.72x) and stayed silent on the version that aggregates `orders` to
one row per customer before joining (2,000 -> 2,000), which also returned the correct $812.17.

It works because a fan-out always changes the number of rows. It would have caught the $1,242.54 figure before it reached a slide. It also flags an inner join that silently drops rows, which is the opposite mistake. When it fires, the fix is to ask which table has several rows per key and to aggregate that table to the unit of observation before joining. If the row growth is intended, as when the question really is about orders, the check still tells the unit has changed, so the average should be read as a per-order number.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [11]:
region_before = customers["region"].copy()      # kept in Python only, so we can check what was blanked

# Each customer has a 15% chance of a missing region, independent of everything else
is_missing = rng.random(N_CUSTOMERS) < 0.15
customers.loc[is_missing, "region"] = None      # None becomes NULL in DuckDB, as with Lab 11's survey table

# Register again so DuckDB sees the changed table
con.register("customers", customers)

# Confirm in SQL: COUNT(*) counts rows, COUNT(region) counts the non-NULL regions
print(con.sql("""
    SELECT COUNT(*)                                   AS total_rows,
           COUNT(region)                              AS non_null_region,
           COUNT(*) - COUNT(region)                   AS null_region,
           ROUND(100.0 * (COUNT(*) - COUNT(region)) / COUNT(*), 1) AS pct_null
    FROM customers
""").df().to_string(index=False))

# The three groups now: West, East and NULL
print(con.sql("""
    SELECT region, COUNT(*) AS customers
    FROM customers
    GROUP BY region
    ORDER BY region
""").df().to_string(index=False))

 total_rows  non_null_region  null_region  pct_null
       3000             2517          483      16.1
region  customers
  East        836
  West       1681
  None        483


The null region is set for 483 of 3,000 customers (16.1%), each chosen independently with probability 0.15, so the share lands a little above 15% by chance. West now has 1,681 customers (down from 2,000), East has 836 (down from 1,000), and 483 customers are in a NULL group. The table still has 3,000 rows (2,517 + 483), and their orders are untouched. Only the `region` label is missing. NULL means unknown, not "other", so these customers are neither West nor East. They cannot be placed in either group, and a filter such as `region = 'West'` treats them as unknown rather than false, so they match
no region filter. West is still a large sample at 1,681, which matters because the long-tailed order counts make small samples noisy.

### Step 2.2: Lose them, then find them (15 pts)

In [2]:
# What the analyst wrote
wrote = con.sql("""
    SELECT COUNT(*) AS customers
    FROM customers
    WHERE region != 'West'
""").df()["customers"][0]

# What the analyst meant: everyone who is not West, including customers with no region
meant = con.sql("""
    SELECT COUNT(*) AS customers
    FROM customers
    WHERE region != 'West' OR region IS NULL
""").df()["customers"][0]

print(f"WHERE region != 'West':                      {wrote:,} rows")
print(f"WHERE region != 'West' OR region IS NULL:    {meant:,} rows")
print(f"Difference:                                  {meant - wrote:,} rows")

# The difference should be exactly the NULL group from Step 2.1
n_null = con.sql("SELECT COUNT(*) - COUNT(region) FROM customers").fetchone()[0]
print(f"NULL-region customers (Step 2.1):            {n_null:,}")
print("Difference equals the NULL count:", meant - wrote == n_null)

`WHERE region != 'West'` returned 836 customers. Adding `OR region IS NULL` returned 1,319. The difference is 483 customers, exactly the NULL group.

For a customer with a NULL region, `region != 'West'` evaluates to NULL
(unknown), not true, and `WHERE` keeps only the rows where the condition is true, so every customer with an unknown region is silently dropped.

The 483 customers only disapear from the query, and they are dropped from `= 'West'` as well as `!= 'West'`. So every regional number is computed only on the 2,517 customers whose region is known. The West and non-West groups (1,681 + 836) add up to 2,517, not 3,000, and nothing in the output says 16% of customers are missing. If those customers were missing at random, the comparisons would be noisier but not biased. If they are systematically different, the comparison is biased, because the dropped customers are not a random sample. The query keeps the customers who were onboarded after the field became mandatory, and the regional figures describe only those. Average revenue would be overstated if the missing customers are smaller, retention would look better than it is if they churn more, and every region would look younger or older than it really is. It would do this in every report that filters on region, with plausible-looking numbers and no error. In this simulation the NULLs were assigned at random, so the regional averages are not biased and only the sample sizes shrink. The bias in the question appears only when missingness is related to the outcome, and the query cannot reveal which case is applicable.

### Step 2.3: Aggregate without collapsing (10 pts)

In [12]:
windowed = con.sql("""
    SELECT
        customer_id,
        region,
        revenue,
        ROUND(AVG(revenue) OVER (PARTITION BY region), 2)           AS region_avg,
        ROUND(revenue - AVG(revenue) OVER (PARTITION BY region), 2) AS vs_region
    FROM customers
    ORDER BY customer_id
""").df()

print(f"Rows in: {len(customers):,}   rows out: {len(windowed):,}")
print("Row count unchanged:", len(windowed) == len(customers))
print(windowed.head(8).to_string(index=False))

# One average per region group, NULL included: this should match a GROUP BY
print(con.sql("""
    SELECT region, COUNT(*) AS customers, ROUND(AVG(revenue), 2) AS avg_revenue
    FROM customers
    GROUP BY region
    ORDER BY region
""").df().to_string(index=False))

Rows in: 3,000   rows out: 3,000
Row count unchanged: True
 customer_id region  revenue  region_avg  vs_region
           0   West  1273.15      815.02     458.13
           1   West   823.80      815.02       8.78
           2   West   242.06      815.02    -572.96
           3   West   717.40      815.02     -97.62
           4   None   827.27      791.72      35.55
           5   West  1380.34      815.02     565.32
           6   None   611.88      791.72    -179.84
           7   None   362.51      791.72    -429.21
region  customers  avg_revenue
  East        836       832.31
  West       1681       815.02
  None        483       791.72


One query puts each customer's revenue and their region's average on the
same row: `AVG(revenue) OVER (PARTITION BY region)`, with no join and no subquery. The output has 3,000 rows, the same as `customers`. `GROUP BY` would have collapsed these to one row per region, which loses the individual customers the director wants to compare. A window function computes the same average but attaches it to every row. The `region_avg` values match the `GROUP BY` averages, and `vs_region` is each customer's revenue minus that average, so positive means above their region and negative means below.

One caution is that `PARTITION BY region` treats NULL as its own group, so the 483 customers with no region are compared against the average of other no-region customers, not against West or East. Customer 4 shows as $35.55 above "their region", but that region is just the unknown group. Unlike `WHERE region = 'West'`, the window function does not drop these customers, so the row count is preserved, but a director reading "above their region's average" should know these customers have no real region.

## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [13]:
TRUE_EFFECT = 25
n = N_CUSTOMERS

# The confounder existing account size, drives assignment and revenue
account_size = rng.normal(0, 1, n)

# Bigger accounts are more likely to be assigned to West, which got the programme
west = (account_size + rng.normal(0, 1, n) > 0).astype(int)

# Revenue depends on account size and, a little, on the programme
revenue = 800 + 150 * account_size + TRUE_EFFECT * west + rng.normal(0, 100, n)

sim = pd.DataFrame({"west": west, "account_size": account_size, "revenue": revenue})
con.register("sim", sim)

# Naive comparison, in SQL: West against not-West, nothing else
by_group = con.sql("""
    SELECT west,
           COUNT(*)            AS customers,
           AVG(revenue)        AS avg_revenue,
           AVG(account_size)   AS avg_size
    FROM sim
    GROUP BY west
    ORDER BY west
""").df().set_index("west")
print(by_group.round(2).to_string())

naive = by_group.loc[1, "avg_revenue"] - by_group.loc[0, "avg_revenue"]

# Adjusted comparison: the same comparison holding account size fixed
fit = smf.ols("revenue ~ west + account_size", data=sim).fit()
adjusted = fit.params["west"]
lo, hi = fit.conf_int().loc["west"]

print(f"\n{'estimate':<28}{'effect':>10}")
print("-" * 38)
print(f"{'Truth':<28}{TRUE_EFFECT:>10.2f}")
print(f"{'Naive (West - not West)':<28}{naive:>10.2f}")
print(f"{'Adjusted (+ account size)':<28}{adjusted:>10.2f}")
print(f"\n95% CI for the adjusted estimate: [{lo:.2f}, {hi:.2f}]")
print("CI contains the truth:", lo <= TRUE_EFFECT <= hi)

      customers  avg_revenue  avg_size
west                                  
0          1484       716.61     -0.56
1          1516       913.86      0.60

estimate                        effect
--------------------------------------
Truth                            25.00
Naive (West - not West)         197.24
Adjusted (+ account size)        27.61

95% CI for the adjusted estimate: [18.88, 36.35]
CI contains the truth: True


In this simulation the programme's true effect is $25 per customer. Account size drives both assignment (bigger accounts are more likely to be in West) and revenue (150 per standard deviation), so West customers start with larger accounts. West averages 913.86 and not-West 716.61. The naive comparison, West minus not-West, gives 197.24, which is about 7.9 times the truth. Controlling for account size gives 27.61 (95% CI [18.88, 36.35]), which is close to the truth, and the interval contains 25.

The naive number is the true effect plus the gap in account size times the effect of account size on revenue. Roughly 25 + 150 x 1.16 = $199, close to the 197.24 observed. It credits the programme with revenue that bigger accounts would have produced anyway.

In real data the adjustment is only as good as the claim that account size is the only thing confounding region and revenue, and that cannot be tested in the data. A real analysis would have to argue from how customers were actually assigned.


### Step 3.2: The control that ruins it (13 pts)

In [14]:
sim["tickets"] = 2.0 * sim["west"] + 0.02 * sim["revenue"] + rng.normal(0, 1, n)
con.register("sim", sim)

fit_size    = smf.ols("revenue ~ west + account_size", data=sim).fit()
fit_tickets = smf.ols("revenue ~ west + account_size + tickets", data=sim).fit()

print(f"{'specification':<34}{'estimate':>10}{'truth':>8}")
print("-" * 52)
print(f"{'Naive (West - not West)':<34}{naive:>10.2f}{TRUE_EFFECT:>8.0f}")
print(f"{'Revenue ~ West + size':<34}{fit_size.params['west']:>10.2f}{TRUE_EFFECT:>8.0f}")
print(f"{'Revenue ~ West + size + tickets':<34}{fit_tickets.params['west']:>10.2f}{TRUE_EFFECT:>8.0f}")

lo, hi = fit_tickets.conf_int().loc["west"]
print(f"\n95% CI with tickets added: [{lo:.2f}, {hi:.2f}]")
print("CI contains the truth:", lo <= TRUE_EFFECT <= hi)

# Why: among customers with similar ticket counts, West customers have LOWER revenue
band = (sim["tickets"] - sim["tickets"].median()).abs() < 1
print(con.sql("""
    SELECT west, COUNT(*) AS customers, ROUND(AVG(revenue), 2) AS avg_revenue
    FROM sim
    WHERE ABS(tickets - (SELECT MEDIAN(tickets) FROM sim)) < 1
    GROUP BY west
    ORDER BY west
""").df().to_string(index=False))

specification                       estimate   truth
----------------------------------------------------
Naive (West - not West)               197.24      25
Revenue ~ West + size                  27.61      25
Revenue ~ West + size + tickets       -75.53      25

95% CI with tickets added: [-79.86, -71.21]
CI contains the truth: False
 west  customers  avg_revenue
    0        249       846.23
    1        231       775.77


**To the director:**

I wouldn't adjust for support tickets, even though they do measure engagement. Tickets are not something that makes a customer spend more or less; they are something that happens as a result of the account-management programme and of how much a customer is spending. Customers in the programme file more tickets because they have someone to ask, and big spenders file more tickets because they use the product more. When we compare customers with the same number
of tickets, we are pairing a programme customer, who got some of those tickets just from the programme, against a non-programme customer who needed to spend more to reach the same count. So the programme customers in the comparison are the ones who spent less, and the programme looks like it hurts revenue. In our data the programme adds 25 dollars per customer. The analysis that adds tickets reports that it costs about 76 dollars, and it is confident about it, with a margin of error of only a few dollars. The analysis without tickets, which compares customers of the same account size, gets $27.61, close to the truth. The rule I'd use is to adjust for things that were true about a customer before they were assigned, such as account size, and not for things that came after, such as tickets, because those already contain the effect we are trying to measure.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [17]:
def review_query(con, sql):
    """The model's function from the P.R.I.M.E. prompt, then your corrections:
    returns a list of structured warnings for one SELECT."""

    #AI output based on the original prompt, 11/1/2026, Chat GPT-6

    """ Review a SELECT query and return structured warnings.

    Each warning contains:
      rule:      stable identifier
      severity:  warning or error
      message:   explanation
      evidence:  details supporting the warning

    Note: This is a lightweight reviewer, not a complete SQL
    lineage or semantic analysis engine.
    """
    warnings = []

    def add(rule, message, evidence=None):
        warnings.append({
            "rule": rule,
            "severity": "warning",
            "message": message,
            "evidence": evidence or {}
        })

    # Validate that the query is a SELECT and can be planned.
    if not sql.strip().lower().startswith(("select", "with")):
        raise ValueError("Only SELECT queries are supported.")

    con.execute("EXPLAIN " + sql)

    # ----------------------------------------------------------
    # Rule 1: Join fan-out
    # ----------------------------------------------------------
    # DuckDB's EXPLAIN exposes the join operators and their
    # estimated cardinalities. We inspect the plan rather than
    # executing the entire joined query a second time.
    #
    # Estimated cardinalities are not proof of actual fan-out.
    # A production implementation should inspect actual operator
    # cardinalities from EXPLAIN ANALYZE or use explicit key checks.
    plan = con.execute("EXPLAIN " + sql).fetchdf()

    plan_text = "\n".join(
        str(value)
        for value in plan.to_numpy().flatten()
    )

    if "JOIN" in plan_text.upper():
        add(
            "JOIN_CARDINALITY",
            "The query contains a join. Check whether the join "
            "changes the intended row grain.",
            {"plan_contains_join": True}
        )

    # ----------------------------------------------------------
    # Rule 2: Nullable filters
    # ----------------------------------------------------------
    # SQL uses three-valued logic: TRUE, FALSE, and UNKNOWN.
    # A WHERE clause retains only TRUE. Therefore, filtering a
    # nullable column with ordinary comparisons silently excludes
    # NULL values. This can change the population being measured,
    # not merely the query's style.
    #
    # This simple check is intentionally conservative.
    where_match = re.search(
        r"\bWHERE\b(.*?)(?:\bGROUP\s+BY\b|\bORDER\s+BY\b|$)",
        sql,
        flags=re.IGNORECASE | re.DOTALL
    )

    if where_match:
        where_clause = where_match.group(1)

        comparisons = re.findall(
            r"\b([a-zA-Z_][\w.]*)\s*(=|<>|!=|>|<|>=|<=)\s*",
            where_clause
        )

        for column, operator in comparisons:
            add(
                "NULL_FILTER",
                f"Filter on {column} may exclude NULL values.",
                {
                    "column": column,
                    "operator": operator,
                    "reason": (
                        "A comparison involving NULL evaluates to "
                        "UNKNOWN, and WHERE keeps only TRUE."
                    )
                }
            )

    # ----------------------------------------------------------
    # Rule 3: Aggregates over joined data
    # ----------------------------------------------------------
    # A one-to-many join can duplicate rows from the parent table.
    # SUM(parent.amount), for example, may count the same amount
    # once for every matching child row.
    #
    # A reliable check needs the intended grain and the join keys.
    # The presence of both a JOIN and an aggregate is a useful
    # warning, but it does not prove that the aggregate is wrong.
    has_aggregate = bool(re.search(
        r"\b(SUM|AVG|COUNT|MIN|MAX)\s*\(",
        sql,
        flags=re.IGNORECASE
    ))

    if has_aggregate and "JOIN" in plan_text.upper():
        add(
            "AGGREGATE_FANOUT",
            "An aggregate is computed over a joined result. "
            "Verify that joins do not duplicate the rows being "
            "aggregated.",
            {
                "aggregate_detected": True,
                "join_detected": True
            }
        )

    return warnings

In [18]:
# Run review_query on your Phase 1 and Phase 2 queries.
import re   # correction 1: the model's function uses re but never imports it

queries = {
    "1.2 un-joined (correct)":
        "SELECT AVG(revenue) FROM customers WHERE region = 'West'",
    "1.2 joined (the $1,242)":
        """SELECT AVG(c.revenue) FROM customers c
           LEFT JOIN orders o ON c.customer_id = o.customer_id
           WHERE c.region = 'West'""",
    "1.3 aggregate first (correct)":
        """SELECT AVG(c.revenue) FROM customers c
           LEFT JOIN (SELECT customer_id, COUNT(*) AS n_orders
                      FROM orders GROUP BY customer_id) o
           ON c.customer_id = o.customer_id
           WHERE c.region = 'West'""",
    "2.2 != 'West' (loses NULLs)":
        "SELECT COUNT(*) FROM customers WHERE region != 'West'",
    "2.2 OR IS NULL (correct)":
        "SELECT COUNT(*) FROM customers WHERE region != 'West' OR region IS NULL",
    "2.3 window (no join, no filter)":
        """SELECT customer_id, region, revenue,
                  AVG(revenue) OVER (PARTITION BY region) AS region_avg
           FROM customers""",
}

for label, sql in queries.items():
    found = review_query(con, sql)
    print(f"{label}: {len(found)} warning(s)")
    for w in found:
        print(f"    {w['rule']:<18} {w['evidence']}")

1.2 un-joined (correct): 1 warning(s)
    NULL_FILTER        {'column': 'region', 'operator': '=', 'reason': 'A comparison involving NULL evaluates to UNKNOWN, and WHERE keeps only TRUE.'}
1.2 joined (the $1,242): 3 warning(s)
    JOIN_CARDINALITY   {'plan_contains_join': True}
    NULL_FILTER        {'column': 'c.region', 'operator': '=', 'reason': 'A comparison involving NULL evaluates to UNKNOWN, and WHERE keeps only TRUE.'}
    AGGREGATE_FANOUT   {'aggregate_detected': True, 'join_detected': True}
1.3 aggregate first (correct): 3 warning(s)
    JOIN_CARDINALITY   {'plan_contains_join': True}
    NULL_FILTER        {'column': 'c.region', 'operator': '=', 'reason': 'A comparison involving NULL evaluates to UNKNOWN, and WHERE keeps only TRUE.'}
    AGGREGATE_FANOUT   {'aggregate_detected': True, 'join_detected': True}
2.2 != 'West' (loses NULLs): 1 warning(s)
    NULL_FILTER        {'column': 'region', 'operator': '!=', 'reason': 'A comparison involving NULL evaluates to UNKNOWN, and 

One change: The AI used re, but did not import it.

**Caught.** The joined query that produced $1,242.54 drew two join-related warnings , so it would have been flagged before it reached a slide. The `!= 'West'` filter that lost 483 customers drew a `NULL_FILTER` warning. The window query was correctly left alone, and it has no join and no filter.

**Missed** The reviewer never measures a row count. It checks whether
the text of the query plan contains the word `JOIN`. It therefore gave the broken join and the correct aggregate-first join the same three
warnings, which is why its alert carries no information about which query is right. The prompt asked for "a join that changes the row count", and the code does not do that. The comment saying fan-out is detected from the plan is an overstatement. The function reads the plan, and a join is present in both queries. Step 1.3 `check_join` separated the two cases and the reviewer could not.

### Task 4.2: The one-page writeup (8 pts)

Dear Director,

The correct average revenue per West customer is 812.17 dollars. The $1,242.54 figure answers revenue per order instead. Revenue is recorded once per customer, but joining to the orders table repeated each customer once for every order, turning 2,000 West customers into 13,440 rows. Customers with many orders also have the highest revenue, so they were counted many times over and the average rose by 53 percent.

The programme works, but modestly. West earns more largely because it has bigger accounts. In my simulation the true effect is $25 per customer, yet the naive West comparison reports 197 dollars, nearly eight times too high. Adjusting for account size gives 27.61 dollars, with a 95 percent interval of 18.88 to 36.35 dollars. Adding support tickets as a control reverses the answer to negative 75.53 dollars, because tickets result from the programme and from revenue rather than causing either. With real data, account size could be argued to be the only confounder but not proven to be.

I recommend that the team compare row counts after every join, aggregating the many-sided table first, as my check did successfully. Missing regions should be handled explicitly, since a plain filter silently dropped 483 customers, or 16 percent. Controls should be limited to facts established before assignment. Automated reviewers should supplement these checks, not replace them, because the AI-written one flagged correct and broken joins alike.


### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* Chat GPT-6

*Prompt:* [Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

*Reply:* Shown above

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.